# Ground Truth Building

This notebook processes every sample in `notebooks/data`, creates machine draft fields through the existing workflow, then treats those suggestions as human-approved final fields inside the notebook.

The approved records are stored in the same SQLite review database used by the UI workflow, and the notebook ends by previewing the MIPROv2 JSONL export records.

## Setup

This cell loads the project modules, configures DSPy from `.env`, and initializes the SQLite review store.

In [ ]:
from pathlib import Path
from pprint import pprint
import json
import mimetypes
import sys

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.core.database import ReviewStore
from app.core.dspy_config import configure_dspy
from app.core.document_processor import process_uploaded_document
from app.core.settings import get_settings

DATA_DIR = NOTEBOOK_DIR / "data"
settings = get_settings()
configure_dspy(settings)
store = ReviewStore(settings.sqlite_db_path, settings.upload_storage_dir)
store.initialize()

print({
    "dspy_model": settings.dspy_model_identifier,
    "db_path": settings.sqlite_db_path,
    "upload_storage_dir": settings.upload_storage_dir,
    "data_files": sorted(path.name for path in DATA_DIR.iterdir() if path.is_file()),
})

## Helpers

The notebook uses these helpers to build a batch, persist uploaded source files, run the extraction workflow, and convert draft fields into final reviewed fields.

In [ ]:
USE_CASE_NAME = "Notebook Ground Truth Bootstrap"
REVIEWED_BY = "notebook_auto_approval"

def guess_content_type(path: Path) -> str | None:
    content_type, _ = mimetypes.guess_type(path.name)
    return content_type

def draft_to_final_fields(workflow_result: dict) -> list[dict]:
    draft_fields = workflow_result.get("consolidated_fields") or workflow_result.get("raw_candidate_fields", [])
    return [
        {
            "field_name": field["proposed_name"],
            "field_value": field["raw_value"],
        }
        for field in draft_fields
        if field.get("proposed_name") and field.get("raw_value")
    ]

def process_file_into_store(batch_id: str, path: Path) -> dict:
    file_bytes = path.read_bytes()
    content_type = guess_content_type(path)
    stored_file_path = store.save_uploaded_file(
        batch_id=batch_id,
        filename=path.name,
        file_bytes=file_bytes,
    )
    result, mime_type, source_type = process_uploaded_document(
        file_bytes=file_bytes,
        filename=path.name,
        content_type=content_type,
    )
    store.add_document(
        batch_id=batch_id,
        workflow_result=result,
        filename=path.name,
        stored_file_path=stored_file_path,
        mime_type=mime_type,
        source_type=source_type,
    )
    return result


## Build Approved Ground Truth Batch

This cell creates a new review batch, processes every file under `notebooks/data`, and immediately submits the generated fields as approved final fields.

In [ ]:
batch_id = store.create_batch(USE_CASE_NAME)
workflow_results = {}
review_submission_documents = []

for path in sorted(DATA_DIR.iterdir()):
    if not path.is_file():
        continue

    result = process_file_into_store(batch_id, path)
    workflow_results[path.name] = result
    review_submission_documents.append(
        {
            "document_id": result["document_id"],
            "final_fields": draft_to_final_fields(result),
        }
    )

store.submit_batch_review(
    batch_id=batch_id,
    reviewed_by=REVIEWED_BY,
    documents=review_submission_documents,
)

print({
    "batch_id": batch_id,
    "reviewed_by": REVIEWED_BY,
    "documents_processed": len(review_submission_documents),
})

## Inspect Stored Batch

This confirms the reviewed records were written into SQLite with final approved fields.

In [ ]:
stored_batch = store.get_batch(batch_id)
print({
    "batch_id": stored_batch["batch_id"],
    "status": stored_batch["status"],
    "use_case_name": stored_batch["use_case_name"],
    "documents": len(stored_batch["documents"]),
})
pprint(stored_batch["documents"][0])

## Preview MIPROv2 Export Records

This reads back the approved records in the same shape exposed by the JSONL export endpoint.

In [ ]:
export_records = store.export_reviewed_records(batch_id=batch_id)
print("record_count", len(export_records))
pprint(export_records[0] if export_records else {})

## Optional: Write a JSONL Snapshot Locally

If you want a file artifact from the notebook run, execute this cell to write the current batch export beside the notebook.

In [ ]:
output_path = NOTEBOOK_DIR / f"{batch_id}_miprov2_export.jsonl"
output_path.write_text("\n".join(json.dumps(record, ensure_ascii=True) for record in export_records), encoding="utf-8")
print(output_path)

## Build DSPy Examples for Optimizer Training

This cell converts the approved MIPROv2 trainset rows into `dspy.Example(...)` objects and marks the `input` field as the model input for optimization.

In [ ]:
import dspy

trainset_records = store.export_miprov2_trainset(batch_id=batch_id)
dspy_examples = [
    dspy.Example(
        input=record["input"],
        ideal_output=record["ideal_output"],
        metadata=record["metadata"],
    ).with_inputs("input")
    for record in trainset_records
]

print("dspy_examples", len(dspy_examples))
dspy_examples[0] if dspy_examples else None